# echo_server — анализ замеров

Ожидаемая раскладка данных:

```
results/summary.csv     
results/raw/<label>.txt  
```

Колонки summary.csv:
`server,rate_target,rate_achieved,msg_size,connections,duration_s,p50,p99,p999,p9999,max` (задержки в ns)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = Path("../results")   
NS = 1000.0                   

plt.rcParams["figure.figsize"] = (9, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


def load_summary() -> pd.DataFrame:
    df = pd.read_csv(RESULTS / "summary.csv")
    df["eff"] = df["rate_achieved"] / df["rate_target"] 
    return df


def load_raw(path: Path) -> np.ndarray:
    return np.sort(np.loadtxt(path, dtype=np.uint64))


def raw_files(pattern: str = "*.txt") -> list[Path]:
    return sorted((RESULTS / "raw").glob(pattern))

## Обзор прогонов

Быстрая проверка, что вообще намерено. Смотри на `eff`: если < ~0.98 — клиент не удержал rate, точка в зоне сатурации, её перцентили описывают отставание генератора, а не сервер.

In [ ]:
df = load_summary()
display(df.sort_values(["server", "msg_size", "rate_target"]))

pivot = df.pivot_table(index="rate_target", columns="server",
                       values="p99", aggfunc="min") / NS
pivot.round(1)

## 1. Latency vs offered load

Главный график: цвет — реализация сервера, стиль линии — перцентиль. У колена переключай `X_COL` на `rate_achieved` — target там врёт.

In [ ]:
MSG_SIZE = 64                     
PCTS = ["p50", "p99", "p999"]
X_COL = "rate_target"         

d = load_summary()
if MSG_SIZE is not None:
    d = d[d["msg_size"] == MSG_SIZE]

fig, ax = plt.subplots()
cmap = plt.get_cmap("tab10")
styles = dict(zip(PCTS, ["-", "--", ":", "-."]))

for i, (server, g) in enumerate(d.groupby("server")):
    g = g.sort_values(X_COL)
    for pct in PCTS:
        ax.plot(g[X_COL], g[pct] / NS, styles[pct],
                marker="o", ms=4, color=cmap(i), label=f"{server} {pct}")

ax.set_xlabel(f"offered load, req/s ({X_COL})")
ax.set_ylabel("latency, \u00b5s")
ax.set_yscale("log")
ax.legend()
ax.set_title(f"latency vs load (msg={MSG_SIZE}B)" if MSG_SIZE else "latency vs load")
plt.show()

## 2. Percentile spectrum (HdrHistogram-style)

Сравнение хвостов на одной точке нагрузки. Глубина «девяток» ограничена размером выборки: для честного p99.99 нужно \u2265 10\u2074 сэмплов.

In [ ]:
FILES = raw_files("*_r40000_m64.txt")
print("files:", [f.name for f in FILES])

fig, ax = plt.subplots()
for path in FILES:
    s = load_raw(path)
    n = len(s)
    max_nines = min(6, int(np.log10(n)))
    pcts = 1.0 - np.logspace(np.log10(0.5), -max_nines, 200)
    idx = np.minimum((pcts * (n - 1)).astype(int), n - 1)
    x = np.log10(1.0 / (1.0 - pcts))    # "количество девяток"
    ax.plot(x, s[idx] / NS, label=f"{path.stem} (n={n})")

ticks = [0.30103, 1, 2, 3, 4, 5, 6]
labels = ["50%", "90%", "99%", "99.9%", "99.99%", "99.999%", "99.9999%"]
ax.set_xticks(ticks)
ax.set_xticklabels(labels)
ax.set_xlim(0, None)
ax.set_xlabel("percentile")
ax.set_ylabel("latency, \u00b5s")
ax.set_yscale("log")
ax.legend()
ax.set_title("latency percentile spectrum")
plt.show()

## 3. CDF

In [ ]:
FILES = raw_files("*_r40000_m64.txt")

fig, ax = plt.subplots()
for path in FILES:
    s = load_raw(path)
    y = np.arange(1, len(s) + 1) / len(s)
    ax.plot(s / NS, y, label=f"{path.stem} (n={len(s)})")

ax.set_xlabel("latency, \u00b5s")
ax.set_xscale("log")
ax.set_ylabel("CDF")
ax.legend()
ax.set_title("latency CDF")
plt.show()

## 4. Прямое сравнение двух прогонов

Например, threaded vs epoll на одной точке — во сколько раз отличаются перцентили.

In [ ]:
A = RESULTS / "raw" / "threaded_r40000_m64.txt"
B = RESULTS / "raw" / "epoll_r40000_m64.txt"

if A.exists() and B.exists():
    sa, sb = load_raw(A), load_raw(B)
    rows = []
    for p in [0.50, 0.90, 0.99, 0.999, 0.9999]:
        qa = np.quantile(sa, p) / NS
        qb = np.quantile(sb, p) / NS
        rows.append({"pct": f"p{p*100:g}", A.stem: qa, B.stem: qb,
                     "ratio": qa / qb})
    display(pd.DataFrame(rows).round(2))
else:
    print("нет пары файлов для сравнения — поправь пути A/B")